# LangGraph Incident Response Workflow

## 1. Project Overview

**Task:** Build a **LangGraph state graph** that walks through an incident response process: collecting logs and context, summarizing the incident, proposing remediation steps, and pausing for human approval before executing or closing.

**Why this matters:** When a production system goes down at 3 AM the on-call engineer must juggle a dozen dashboards, piece together a timeline, and decide on a fix — all under pressure. An LLM-powered graph can automate the tedious parts (log aggregation, timeline construction, writeup) while keeping the human in the loop for critical decisions.

**Pipeline Nodes:**

```
  START
    │
    ▼
  ┌────────────────────┐
  │  collect_context    │  gather logs, metrics, and alert metadata
  └─────────┬──────────┘
            │
            ▼
  ┌────────────────────┐
  │  summarize_incident│  build a structured incident summary
  └─────────┬──────────┘
            │
            ▼
  ┌────────────────────┐
  │  propose_actions    │  suggest remediation steps ranked by risk
  └─────────┬──────────┘
            │
            ▼
  ┌────────────────────┐
  │  approval_gate      │  ★ PAUSE — human reviews & decides
  └─────────┬──────────┘
       conditional
      ┌────┼────────┐
      ▼    ▼        ▼
   approve revise  reject
      │    │        │
      │    └→ back to propose_actions
      │             │
      ▼             ▼
  ┌────────────────────┐
  │  compile_report     │  write the final incident report
  └─────────┬──────────┘
            │
            ▼
          END
```

**Stack:** `LangGraph`, `ChatOllama` + `qwen3.5:9b` (local, no API keys)

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Design a **TypedDict state schema** for an incident response pipeline |
| 2 | Build **single-responsibility nodes** for each phase of the response |
| 3 | Implement a **human-in-the-loop approval gate** that pauses the graph |
| 4 | Use **conditional edges** to branch on human decisions |
| 5 | Add a **revision loop** with a safety cap to prevent infinite cycling |
| 6 | Inspect **intermediate state** after each node for debugging |
| 7 | Understand **state passing** — how partial updates accumulate |

## 3. How This Workflow Differs From a Single Prompt

### The single-prompt approach (fragile)
```
  prompt = "Here are 200 log lines. Summarize the incident,
            propose fixes, and write a report."
  response = llm(prompt)  # one giant blob — no review, no control
```

### The graph approach (robust)
```
  collect_context  → structured log analysis
  summarize        → reviewed summary with timeline
  propose_actions  → ranked actions with risk labels
  approval_gate    → ★ HUMAN CHECK ★ — nothing runs unreviewed
  compile_report   → final document incorporating approval notes
```

**Benefits of the graph:**
- Each step is **inspectable** — you can see the summary before actions are proposed
- The approval gate **prevents automation from acting on bad summaries**
- Revision loops **improve quality** without restarting from scratch
- State is **serializable** — you can save and resume across sessions

## 4. State Passing — How Partial Updates Work

```
  ┌─────────────────────────────────────────────────────────────────┐
  │              STATE PASSING IN THIS PIPELINE                     │
  ├─────────────────────────────────────────────────────────────────┤
  │                                                                 │
  │  After collect_context:                                        │
  │    state = {incident_input: {...}, context_report: "...",       │
  │             summary: "", actions: "", decision: "", ...}        │
  │                                                                 │
  │  After summarize_incident:                                     │
  │    state = {incident_input: {...}, context_report: "...",       │
  │             summary: "Timeline: ...", actions: "", ...}        │
  │         ← only summary changed; everything else preserved      │
  │                                                                 │
  │  After propose_actions:                                        │
  │    state = {..., summary: "Timeline: ...",                      │
  │             actions: "1. Restart pod...", ...}                  │
  │         ← only actions changed                                 │
  │                                                                 │
  │  KEY: Nodes return ONLY their outputs.                         │
  │  LangGraph merges into the full state automatically.            │
  └─────────────────────────────────────────────────────────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langgraph

print("Dependencies: langchain, langgraph")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict
from datetime import datetime

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

LLM_MODEL = "qwen3.5:9b"

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"LangGraph: StateGraph, START, END")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.2) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 100):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Simulated Incident Data & State Schema

## 8. Simulated Incidents

In production these would come from PagerDuty, Datadog, or CloudWatch. Here we use realistic simulated incidents with logs, metrics, and alert metadata.

In [ ]:
INCIDENTS = [
    {
        "incident_id": "INC-2024-0847",
        "severity": "P1",
        "title": "Payment service returning 500s — checkout flow blocked",
        "triggered_at": "2024-11-15T03:22:00Z",
        "service": "payment-gateway",
        "alert_source": "Datadog",
        "on_call": "Jamie Torres",
        "logs": [
            "03:20:12 [payment-gateway] INFO  Health check passed",
            "03:21:45 [payment-gateway] WARN  Connection pool exhausted — waiting for free connection",
            "03:21:47 [payment-gateway] WARN  DB query timeout after 30s: SELECT * FROM transactions WHERE ...",
            "03:21:48 [payment-gateway] ERROR ConnectionPool: 0/20 available, 20/20 active, 47 pending",
            "03:21:49 [payment-gateway] ERROR Payment.process() failed: org.postgresql.util.PSQLException: Connection pool timeout",
            "03:21:50 [nginx] ERROR 502 Bad Gateway — upstream payment-gateway:8080 timed out",
            "03:21:51 [payment-gateway] ERROR Circuit breaker OPEN for postgres-primary",
            "03:22:00 [datadog] ALERT P1: payment-gateway error rate > 50% (current: 87%)",
            "03:22:15 [payment-gateway] ERROR Retry exhausted for transaction txn_abc123",
            "03:22:30 [checkout-frontend] ERROR /api/checkout returned 500 — showing error page to user",
            "03:23:00 [datadog] ALERT: checkout success rate dropped from 99.2% to 12.4%",
            "03:25:00 [postgres-primary] WARN  Long-running query detected: 45s, pid=12847",
            "03:25:01 [postgres-primary] WARN  Lock wait timeout: table=transactions, pid=12847 blocked by pid=10234",
            "03:25:30 [postgres-primary] INFO  Autovacuum running on table 'transactions' (142M dead tuples)",
        ],
        "metrics": {
            "error_rate": "87%",
            "p99_latency": "32s (normal: 200ms)",
            "active_db_connections": "20/20 (pool exhausted)",
            "pending_requests": 47,
            "checkout_success_rate": "12.4% (normal: 99.2%)",
            "affected_users_estimate": "~2,400 in the last 10 min",
        },
        "recent_changes": [
            "14:00 yesterday — deployed payment-gateway v2.14.3 (added batch reconciliation query)",
            "No infra changes in 48h",
        ],
    },
    {
        "incident_id": "INC-2024-0912",
        "severity": "P2",
        "title": "Search service degraded — results returning stale data",
        "triggered_at": "2024-11-18T14:05:00Z",
        "service": "search-api",
        "alert_source": "Grafana",
        "on_call": "Alex Kim",
        "logs": [
            "14:00:00 [search-api] INFO  Index refresh started",
            "14:02:30 [search-api] WARN  Elasticsearch cluster health: YELLOW — 2 unassigned shards",
            "14:03:00 [search-api] WARN  Index refresh taking longer than expected: 180s (threshold: 60s)",
            "14:04:00 [search-api] ERROR Elasticsearch query timeout: index=products, took=15s",
            "14:04:15 [search-api] WARN  Falling back to stale cache (age: 47min)",
            "14:05:00 [grafana] ALERT P2: search-api cache_hit_rate > 95% (serving stale data)",
            "14:06:00 [elasticsearch] WARN  Node es-data-03 disk usage at 92% — watermark exceeded",
            "14:06:30 [elasticsearch] INFO  Shard relocation triggered: moving 3 shards from es-data-03",
            "14:08:00 [search-api] ERROR New product listings not appearing in search results",
        ],
        "metrics": {
            "cache_hit_rate": "97% (normal: 40%)",
            "index_freshness": "47 min stale (normal: <5min)",
            "es_cluster_health": "YELLOW",
            "disk_usage_es_data_03": "92%",
            "affected_queries": "~800/min returning stale results",
        },
        "recent_changes": [
            "12:00 today — product catalog bulk import (85K new SKUs)",
            "Yesterday — Elasticsearch minor version upgrade 8.11.1 → 8.11.3",
        ],
    },
]

print(f"Simulated incidents: {len(INCIDENTS)}")
for inc in INCIDENTS:
    print(f"  {inc['incident_id']} [{inc['severity']}] {inc['title'][:60]}")
    print(f"    Service: {inc['service']} | Logs: {len(inc['logs'])} lines")

## 9. State Schema

Every field has a clear **owner** (the node that writes it) and **readers** (nodes that consume it).

| Field | Written By | Read By |
|-------|-----------|--------|
| `incident_input` | Input | collect_context |
| `context_report` | collect_context | summarize_incident |
| `incident_summary` | summarize_incident | propose_actions, compile_report |
| `proposed_actions` | propose_actions | approval_gate |
| `decision` | approval_gate | routing function |
| `approval_notes` | approval_gate | compile_report |
| `revision_feedback` | approval_gate | propose_actions (on revision) |
| `revision_count` | approval_gate | routing function (safety limit) |
| `final_report` | compile_report | caller |
| `current_node` | every node | debugging |

In [ ]:
class IncidentState(TypedDict):
    """State schema for the incident response pipeline.

    Design rationale:
    - incident_input holds the raw alert + logs (immutable after init)
    - context_report is the structured analysis of logs and metrics
    - incident_summary is the human-readable timeline and impact assessment
    - proposed_actions are ranked remediation steps
    - decision is the human verdict: approve / revise / reject
    - revision_feedback carries improvement instructions for the revise loop
    - revision_count prevents infinite loops
    - final_report is the compiled incident report (only on approve/reject)
    """
    incident_input: dict
    context_report: str
    incident_summary: str
    proposed_actions: str
    decision: str
    approval_notes: str
    revision_feedback: str
    revision_count: int
    final_report: str
    current_node: str


print("State schema: IncidentState")
for name, typ in IncidentState.__annotations__.items():
    print(f"  {name}: {typ}")

---

# Part B — Build the Graph Nodes

## 10. Node 1: Collect Context

This node takes the raw incident (logs, metrics, alert metadata) and produces a **structured context report**. It identifies the timeline, correlates events, and flags anomalies.

**Why a dedicated node?** Raw logs are noisy. The rest of the pipeline needs a clean, structured view — not 14 raw log lines.

In [ ]:
CONTEXT_SYSTEM = """You are an SRE log analyst. Given raw incident data (alert, logs, metrics, recent changes),
produce a STRUCTURED CONTEXT REPORT with these sections:

1. TIMELINE: chronological list of key events with timestamps
2. ROOT CAUSE INDICATORS: log patterns and metrics that point to the cause
3. BLAST RADIUS: what services, users, and flows are affected
4. CORRELATED CHANGES: recent deployments or config changes that may be related

Be concise and factual. Cite specific log lines and metric values."""

CONTEXT_PROMPT = """INCIDENT: {incident_id} [{severity}] — {title}
Service: {service}
Triggered: {triggered_at}
Alert Source: {alert_source}
On-Call: {on_call}

LOGS:
{logs}

METRICS:
{metrics}

RECENT CHANGES:
{changes}

Produce the structured context report."""


def collect_context(state: IncidentState) -> dict:
    """Node: Analyze raw logs/metrics into a structured context report."""
    print("  [NODE] collect_context")
    inc = state["incident_input"]

    report = ask(
        CONTEXT_PROMPT.format(
            incident_id=inc["incident_id"],
            severity=inc["severity"],
            title=inc["title"],
            service=inc["service"],
            triggered_at=inc["triggered_at"],
            alert_source=inc["alert_source"],
            on_call=inc["on_call"],
            logs="\n".join(inc["logs"]),
            metrics=json.dumps(inc["metrics"], indent=2),
            changes="\n".join(inc["recent_changes"]),
        ),
        system=CONTEXT_SYSTEM,
        temperature=0.1,
    )
    print(f"    Context report: {len(report)} chars")
    return {"context_report": report, "current_node": "collect_context"}


print("Node defined: collect_context")
print("  READS:  incident_input")
print("  WRITES: context_report")

## 11. Node 2: Summarize Incident

Turns the structured context into a concise **incident summary** — the kind you'd post in Slack or a war-room channel.

In [ ]:
SUMMARY_SYSTEM = """Write a concise incident summary for an engineering war room. Include:
1. ONE-LINE SUMMARY: what happened, in plain language
2. IMPACT: which users/flows are affected and estimated scope
3. ROOT CAUSE (suspected): most likely cause based on the evidence
4. CURRENT STATUS: is it ongoing, mitigated, or resolved?
Keep it under 200 words. Write for engineers, not management."""


def summarize_incident(state: IncidentState) -> dict:
    """Node: Create a war-room-style incident summary."""
    print("  [NODE] summarize_incident")
    inc = state["incident_input"]

    summary = ask(
        f"INCIDENT: {inc['incident_id']} [{inc['severity']}] — {inc['title']}\n\n"
        f"CONTEXT REPORT:\n{state['context_report'][:2000]}\n\n"
        f"Write the war-room incident summary.",
        system=SUMMARY_SYSTEM,
        temperature=0.1,
    )
    print(f"    Summary: {len(summary)} chars")
    return {"incident_summary": summary, "current_node": "summarize_incident"}


print("Node defined: summarize_incident")
print("  READS:  incident_input, context_report")
print("  WRITES: incident_summary")

## 12. Node 3: Propose Actions

Suggests concrete remediation steps, each labeled with a **risk level** and **estimated time**. If this is a revision round, the node incorporates feedback from the human reviewer.

**Why rank by risk?** During an incident, the team should try the lowest-risk fix first. Restarting a pod is safer than running a database migration.

In [ ]:
ACTIONS_SYSTEM = """Propose remediation actions for a production incident.
For each action, specify:
- STEP NUMBER and DESCRIPTION
- RISK LEVEL: low / medium / high
- ESTIMATED TIME: how long it takes
- ROLLBACK PLAN: how to undo if it makes things worse

Order actions from lowest risk to highest risk.
Include 3-5 actions. Be specific — name actual commands, configs, or procedures."""

ACTIONS_PROMPT = """INCIDENT SUMMARY:
{summary}

CONTEXT REPORT (key details):
{context}

{revision_section}

Propose 3-5 remediation actions, ordered by risk (lowest first)."""


def propose_actions(state: IncidentState) -> dict:
    """Node: Suggest ranked remediation steps."""
    print("  [NODE] propose_actions")

    revision_section = ""
    if state.get("revision_feedback") and state.get("revision_count", 0) > 0:
        revision_section = (
            f"REVIEWER FEEDBACK (incorporate these changes):\n"
            f"{state['revision_feedback']}\n\n"
            f"PREVIOUS PROPOSAL (improve, don't start over):\n"
            f"{state['proposed_actions'][:800]}"
        )
        print(f"    Revision round {state['revision_count']} — incorporating feedback")

    actions = ask(
        ACTIONS_PROMPT.format(
            summary=state["incident_summary"][:1000],
            context=state["context_report"][:1000],
            revision_section=revision_section,
        ),
        system=ACTIONS_SYSTEM,
        temperature=0.3,
    )
    print(f"    Actions proposed: {len(actions)} chars")
    return {"proposed_actions": actions, "current_node": "propose_actions"}


print("Node defined: propose_actions")
print("  READS:  incident_summary, context_report, revision_feedback (if revision)")
print("  WRITES: proposed_actions")

## 13. Node 4: Approval Gate (Human-in-the-Loop)

This is the **critical control point**. In a real system this would pause and wait for a Slack reaction, a PagerDuty acknowledgment, or a web form submission. Here we simulate the human decision.

### How the approval gate works

```
  ┌──────────────────────────────────────────────────────┐
  │               APPROVAL GATE LOGIC                    │
  ├──────────────────────────────────────────────────────┤
  │                                                      │
  │  1. Display the incident summary and proposed plan   │
  │  2. ★ PAUSE — wait for human input ★                │
  │     (simulated via pre-defined decisions here)       │
  │  3. Record the decision + any notes                  │
  │  4. Return to the graph for routing                  │
  │                                                      │
  │  Decisions:                                          │
  │    approve → proceed to compile_report               │
  │    revise  → loop back to propose_actions            │
  │    reject  → skip to compile_report (mark rejected)  │
  └──────────────────────────────────────────────────────┘
```

**In production**, you'd replace the simulated decision with:
- `input()` for interactive notebooks
- A webhook callback for async workflows
- LangGraph's built-in `interrupt()` / checkpointing

In [ ]:
# Pre-defined human decisions for reproducibility
# Format: (decision, notes, feedback_if_revise)
SIMULATED_DECISIONS = {
    # First incident: first review requests revision, second approves
    "INC-2024-0847": [
        (
            "revise",
            "Good start, but the plan needs a database-specific step.",
            "Add a step to kill the long-running query (pid 12847) before restarting the service. "
            "Also add monitoring: watch the connection pool count after each step.",
        ),
        (
            "approve",
            "Plan looks solid now. Proceed with step 1 first, escalate if no improvement in 5 min.",
            "",
        ),
    ],
    # Second incident: approved on first review
    "INC-2024-0912": [
        (
            "approve",
            "Straightforward disk issue. Approved — start with shard relocation.",
            "",
        ),
    ],
}


def approval_gate(state: IncidentState) -> dict:
    """Node: Simulate human review of proposed actions.

    In production, this would PAUSE the graph and wait for human input.
    Here we use pre-defined decisions for reproducibility.
    """
    print("  [NODE] approval_gate")
    inc_id = state["incident_input"]["incident_id"]
    rev = state.get("revision_count", 0)

    # Display what the human would see
    print("  " + "=" * 50)
    print("  ★ HUMAN REVIEW CHECKPOINT ★")
    print(f"  Incident: {inc_id}")
    print(f"  Review round: {rev + 1}")
    print("  " + "=" * 50)
    print("  Summary:")
    for line in state["incident_summary"][:300].split("\n")[:5]:
        print(f"    {line}")
    print("  Proposed actions:")
    for line in state["proposed_actions"][:400].split("\n")[:8]:
        print(f"    {line}")
    print("  " + "-" * 50)

    # Get the simulated decision
    decisions = SIMULATED_DECISIONS.get(inc_id, [("approve", "Auto-approved", "")])
    idx = min(rev, len(decisions) - 1)
    decision, notes, feedback = decisions[idx]

    print(f"  Decision: {decision.upper()}")
    print(f"  Notes: {notes}")
    if feedback:
        print(f"  Revision feedback: {feedback[:80]}...")

    return {
        "decision": decision,
        "approval_notes": notes,
        "revision_feedback": feedback,
        "revision_count": rev + 1,
        "current_node": "approval_gate",
    }


print("Node defined: approval_gate")
print("  READS:  incident_input, incident_summary, proposed_actions")
print("  WRITES: decision, approval_notes, revision_feedback, revision_count")
print(f"  Simulated decisions configured for {len(SIMULATED_DECISIONS)} incidents")

## 14. Node 5: Compile Report

After the plan is approved (or rejected), compile everything into a structured incident report.

In [ ]:
REPORT_SYSTEM = """Write a formal incident report. Include:
1. INCIDENT HEADER (ID, severity, service, time)
2. EXECUTIVE SUMMARY (2-3 sentences)
3. TIMELINE (chronological events)
4. ROOT CAUSE ANALYSIS
5. REMEDIATION PLAN (the approved actions)
6. FOLLOW-UP ITEMS (preventive measures for the future)
Use Markdown formatting. Be thorough but concise."""


def compile_report(state: IncidentState) -> dict:
    """Node: Compile the final incident report."""
    print("  [NODE] compile_report")
    inc = state["incident_input"]

    status = "APPROVED" if state["decision"] == "approve" else "REJECTED"

    report = ask(
        f"INCIDENT: {inc['incident_id']} [{inc['severity']}] — {inc['title']}\n\n"
        f"CONTEXT:\n{state['context_report'][:800]}\n\n"
        f"SUMMARY:\n{state['incident_summary'][:500]}\n\n"
        f"APPROVED ACTIONS ({status}):\n{state['proposed_actions'][:800]}\n\n"
        f"REVIEWER NOTES: {state['approval_notes']}\n\n"
        f"Revision rounds: {state['revision_count']}\n\n"
        f"Write the formal incident report.",
        system=REPORT_SYSTEM,
        temperature=0.1,
    )

    # Prepend metadata header
    header = (
        f"{'=' * 60}\n"
        f"INCIDENT REPORT — {inc['incident_id']}\n"
        f"Status: {status}\n"
        f"Reviewer: {inc['on_call']}\n"
        f"Revision rounds: {state['revision_count']}\n"
        f"{'=' * 60}\n\n"
    )
    final = header + report
    print(f"    Report compiled: {len(final)} chars")
    return {"final_report": final, "current_node": "compile_report"}


print("Node defined: compile_report")
print("  READS:  incident_input, context_report, incident_summary,")
print("          proposed_actions, decision, approval_notes")
print("  WRITES: final_report")

---

# Part C — Conditional Routing & Graph Assembly

## 15. Routing After Approval Gate

Three paths:
- **approve** → `compile_report`
- **revise** (under cap) → loop back to `propose_actions`
- **revise** (at cap) or **reject** → `compile_report` (with appropriate status)

In [ ]:
MAX_REVISIONS = 3


def route_after_approval(state: IncidentState) -> str:
    """Conditional edge: route based on human decision."""
    decision = state.get("decision", "approve")
    revisions = state.get("revision_count", 0)

    if decision == "approve":
        print(f"    [ROUTE] approved → compile_report")
        return "compile_report"

    if decision == "reject":
        print(f"    [ROUTE] rejected → compile_report (will record rejection)")
        return "compile_report"

    # decision == "revise"
    if revisions >= MAX_REVISIONS:
        print(f"    [ROUTE] max revisions ({MAX_REVISIONS}) hit → forcing compile_report")
        return "compile_report"

    print(f"    [ROUTE] revise → propose_actions (round {revisions})")
    return "propose_actions"


print(f"Routing function: route_after_approval (max {MAX_REVISIONS} revisions)")
print("  approve      → compile_report")
print("  reject       → compile_report")
print("  revise (ok)  → propose_actions")
print("  revise (cap) → compile_report")

## 16. Assemble the StateGraph

Wiring the graph. Watch how each `add_edge` mirrors the architecture diagram from Section 1.

In [ ]:
workflow = StateGraph(IncidentState)

# Register all five nodes
workflow.add_node("collect_context", collect_context)
workflow.add_node("summarize_incident", summarize_incident)
workflow.add_node("propose_actions", propose_actions)
workflow.add_node("approval_gate", approval_gate)
workflow.add_node("compile_report", compile_report)

# Linear edges: START → collect → summarize → propose → approval
workflow.add_edge(START, "collect_context")
workflow.add_edge("collect_context", "summarize_incident")
workflow.add_edge("summarize_incident", "propose_actions")
workflow.add_edge("propose_actions", "approval_gate")

# Conditional edge: approval decision routes the graph
workflow.add_conditional_edges(
    "approval_gate",
    route_after_approval,
    {
        "compile_report": "compile_report",
        "propose_actions": "propose_actions",
    },
)

# Terminal edge
workflow.add_edge("compile_report", END)

# Compile
app = workflow.compile()

print("Graph compiled!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print()
print("Flow:")
print("  START → collect_context → summarize_incident → propose_actions → approval_gate")
print("  approval_gate ──(approve)──→ compile_report → END")
print("  approval_gate ──(reject)───→ compile_report → END")
print("  approval_gate ──(revise)───→ propose_actions (loop)")

In [ ]:
# Visualize graph structure
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering not available: {e}")
    print("\nGraph (text):")
    print("  __start__ --> collect_context --> summarize_incident")
    print("  summarize_incident --> propose_actions --> approval_gate")
    print("  approval_gate --(approve/reject)--> compile_report --> __end__")
    print("  approval_gate --(revise)--> propose_actions")

---

# Part D — Execute the Pipeline

## 17. Run on Incident 1 (Payment Service — with revision)

In [ ]:
def make_initial_state(incident: dict) -> IncidentState:
    return {
        "incident_input": incident,
        "context_report": "",
        "incident_summary": "",
        "proposed_actions": "",
        "decision": "",
        "approval_notes": "",
        "revision_feedback": "",
        "revision_count": 0,
        "final_report": "",
        "current_node": "start",
    }


print(f"Running pipeline: {INCIDENTS[0]['incident_id']} — {INCIDENTS[0]['title'][:50]}")
print("This incident has a REVISE decision on first review.")
print("Expected flow: collect → summarize → propose → ★revise★ → propose → ★approve★ → report")
print("=" * 70)

result_1 = app.invoke(
    make_initial_state(INCIDENTS[0]),
    {"recursion_limit": 25},
)

print(f"\nPipeline complete.")
print(f"  Decision: {result_1['decision']}")
print(f"  Revision rounds: {result_1['revision_count']}")
print(f"  Report length: {len(result_1['final_report'])} chars")

## 18. Inspect Intermediate State

The graph's key advantage: every intermediate output is visible.

In [ ]:
print("STATE INSPECTION — Incident 1")
print("=" * 80)

print("\n[1] CONTEXT REPORT (first 400 chars):")
wrap_print(result_1["context_report"][:400])

print("\n" + "-" * 80)
print("[2] INCIDENT SUMMARY:")
wrap_print(result_1["incident_summary"][:500])

print("\n" + "-" * 80)
print("[3] PROPOSED ACTIONS (final version after revision):")
wrap_print(result_1["proposed_actions"][:600])

print("\n" + "-" * 80)
print(f"[4] DECISION: {result_1['decision'].upper()}")
print(f"    Approval notes: {result_1['approval_notes']}")
print(f"    Revisions: {result_1['revision_count']}")

## 19. Step-by-Step Streaming View

Using `app.stream()` shows each node firing in sequence — including the revision loop.

In [ ]:
print("STREAMING EXECUTION — Incident 1 (with revision loop)")
print("=" * 60)

step = 0
for chunk in app.stream(make_initial_state(INCIDENTS[0]), {"recursion_limit": 25}):
    step += 1
    for node_name, node_output in chunk.items():
        keys_written = [k for k in node_output.keys() if k != "current_node"]
        sizes = {k: len(str(node_output[k])) for k in keys_written}
        print(f"  Step {step}: {node_name:<22} → wrote {keys_written}")
        for k, v in sizes.items():
            print(f"       {k}: {v} chars")

print(f"\nTotal steps: {step}")
print("Notice steps 5-6: propose_actions runs twice due to the revision loop.")

## 20. Run on Incident 2 (Search Service — approved first time)

In [ ]:
print(f"Running pipeline: {INCIDENTS[1]['incident_id']} — {INCIDENTS[1]['title'][:50]}")
print("This incident is approved on first review — no revision loop.")
print("Expected flow: collect → summarize → propose → ★approve★ → report")
print("=" * 70)

result_2 = app.invoke(
    make_initial_state(INCIDENTS[1]),
    {"recursion_limit": 25},
)

print(f"\nPipeline complete.")
print(f"  Decision: {result_2['decision']}")
print(f"  Revision rounds: {result_2['revision_count']}")
print(f"  Report length: {len(result_2['final_report'])} chars")

---

# Part E — View Final Reports

## 21. Incident Reports

In [ ]:
all_results = [result_1, result_2]

for i, (result, inc) in enumerate(zip(all_results, INCIDENTS), 1):
    print(f"\n{'#' * 80}")
    print(f"# INCIDENT {i}: {inc['incident_id']} — {inc['title'][:50]}")
    print(f"{'#' * 80}")
    wrap_print(result["final_report"])
    print()

---

# Part F — Pipeline Analysis

## 22. Cross-Incident Metrics

In [ ]:
print("PIPELINE METRICS")
print("=" * 80)
print(f"{'Incident':<18} {'Severity':<10} {'Decision':<12} {'Revisions':<12} {'Report Len':<12}")
print("-" * 64)

for r, inc in zip(all_results, INCIDENTS):
    print(f"{inc['incident_id']:<18} {inc['severity']:<10} {r['decision']:<12} "
          f"{r['revision_count']:<12} {len(r['final_report']):<12}")

avg_rev = sum(r["revision_count"] for r in all_results) / len(all_results)
approved = sum(1 for r in all_results if r["decision"] == "approve")
print(f"\n  Approval rate:     {approved}/{len(all_results)}")
print(f"  Average revisions: {avg_rev:.1f}")

## 23. State Accumulation Visualization

In [ ]:
print("STATE ACCUMULATION — Incident 1")
print("=" * 60)

r = all_results[0]
fields = [
    ("incident_input", json.dumps(r["incident_input"])),
    ("context_report", r["context_report"]),
    ("incident_summary", r["incident_summary"]),
    ("proposed_actions", r["proposed_actions"]),
    ("approval_notes", r["approval_notes"]),
    ("final_report", r["final_report"]),
]

cumulative = 0
for name, content in fields:
    size = len(content)
    cumulative += size
    bar = "█" * min(size // 80, 40)
    print(f"  {name:<22} {size:>5} chars  {bar}")

print(f"\n  Total state: {cumulative:,} chars")
print("  The state grows as each node adds its output.")
print("  No node ever overwrites another node's fields.")

## 24. Workflow Recap & Node Design

```
  Node                 READS                          WRITES
  ──────────────────── ────────────────────────────── ────────────────────────
  collect_context      incident_input                 context_report
  summarize_incident   incident_input, context_report incident_summary
  propose_actions      incident_summary, context_report, proposed_actions
                       revision_feedback (if revision)
  approval_gate        incident_input, incident_summary, decision, approval_notes,
                       proposed_actions               revision_feedback,
                                                      revision_count
  compile_report       ALL fields                     final_report

  DESIGN RULES APPLIED:
  1. Single responsibility — each node does ONE thing
  2. Partial updates — nodes return ONLY changed fields
  3. Stateless execution — all data lives in the graph state
  4. Explicit ownership — each field has ONE writer
```

## 25. The Approval Gate Pattern

The approval gate is the most important pattern in production LLM workflows:

| Aspect | Why It Matters |
|--------|---------------|
| **Prevents unreviewed automation** | LLMs hallucinate — a human checks before action |
| **Enables revision loops** | Feedback improves proposals without restarting |
| **Creates an audit trail** | Every decision, note, and revision is recorded in state |
| **Supports async workflows** | In production, the gate can wait hours for a human |
| **Safety cap prevents infinite loops** | `MAX_REVISIONS` stops pathological cycles |

### Implementing real approval gates

```python
# Option 1: Interactive notebook
decision = input("approve / revise / reject: ")

# Option 2: LangGraph checkpointing
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer, interrupt_before=["approval_gate"])
# Graph pauses before approval_gate; resume with updated state

# Option 3: Webhook callback (production)
# Post to Slack → wait for reaction → resume graph via API
```

## 26. Production Extensions

1. **Real log ingestion** — connect to Datadog, CloudWatch, or Loki APIs in `collect_context`
2. **Runbook lookup** — add a node that searches a runbook database for known fixes
3. **Auto-remediation** — after approval, add an `execute_actions` node that runs safe commands
4. **Escalation routing** — if severity is P0 or revisions exceed threshold, page a manager
5. **Post-incident review** — after resolution, generate a blameless postmortem template

## 27. Exercises

### Exercise 1: Interactive Approval
Replace the simulated decisions with `input()` so you can interactively approve, revise, or reject proposals. Add a text field where you can type revision feedback.

### Exercise 2: Severity-Based Routing
Add a `triage` node before `collect_context` that assigns a severity (P0-P3) based on the alert metadata. For P0 incidents, skip the approval gate and auto-approve to save time.

### Exercise 3: Runbook Integration
Create a `lookup_runbook` node between `summarize_incident` and `propose_actions`. It should match the incident against a dictionary of known issues and include relevant runbook steps in the proposed actions.

### Mini Challenge: Multi-Incident Coordination
Build a wrapper that detects when two incidents affect related services (e.g., both involve the database), correlates them, and produces a combined remediation plan.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Incidents processed:  {len(all_results)}")
print(f"Approval rate:        {approved}/{len(all_results)}")
print(f"Avg revisions:        {avg_rev:.1f}")
print()
print("Graph topology:")
print("  START → collect_context → summarize_incident → propose_actions")
print("  → approval_gate ──(approve)──→ compile_report → END")
print("  → approval_gate ──(reject)───→ compile_report → END")
print("  → approval_gate ──(revise)───→ propose_actions (loop)")
print()
print("Nodes built:")
print("  1. collect_context     — structures raw logs/metrics into a report")
print("  2. summarize_incident  — produces a war-room incident summary")
print("  3. propose_actions     — suggests ranked remediation steps")
print("  4. approval_gate       — ★ human review with approve/revise/reject")
print("  5. compile_report      — writes the formal incident report")
print()
print("Key patterns:")
print("  • Human-in-the-loop approval gate with revision loop")
print("  • Conditional routing: approve / revise / reject")
print("  • TypedDict state with documented field ownership")
print("  • Partial state updates — nodes return only changed fields")
print("  • Revision cap prevents infinite loops")
print("  • Step-by-step streaming for observability")

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Separate context gathering from analysis** — raw logs need structured extraction before summarization |
| 2 | **The approval gate is non-negotiable** — never auto-execute LLM-proposed remediation without human review |
| 3 | **Revision loops improve quality** — human feedback makes the second proposal significantly better |
| 4 | **Cap your loops** — `MAX_REVISIONS` prevents infinite cycling when the LLM and reviewer disagree |
| 5 | **State is your audit trail** — every intermediate output is inspectable and serializable |
| 6 | **Nodes should be stateless** — all data lives in the graph state, making the pipeline resumable |
| 7 | **Streaming shows the revision loop** — `app.stream()` makes it visible when a node runs twice |
| 8 | **Risk-ranked actions help the on-call** — try the safest fix first, escalate only if needed |
| 9 | **The graph IS the runbook** — the pipeline encodes institutional knowledge about incident response |
| 10 | **Production needs real pause** — replace simulated decisions with LangGraph checkpoints or webhooks |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*